# Train a LoRA adapter on Qwen3 8B

This notebook fine-tunes a small LoRA adapter on Qwen3 8B with QLoRA.

Training reads the merged JSONL, keeps deterministic train/val/test splits, and runs one full epoch over the training split by default.

On Colab, complete Hugging Face checkpoints are mirrored to Google Drive every 50 optimizer steps. A restarted session restores the latest compatible checkpoint automatically.


## Public training dataset

The training JSONL is hosted on Google Drive so tutors can inspect or rerun training without rebuilding the pipeline:

- Public dataset link: [research_lora_train.jsonl](https://drive.google.com/file/d/18ZFPdpf5114kO_nSNcYYyiePGzRrFxVT/view?usp=sharing)
- Notebook constant: `PUBLIC_TRAIN_JSONL_URL`
- Expected file name: `research_lora_train.jsonl`
- Expected row count: 17,298 chat-format instruction rows
- Guardrail: training stops before model load unless exactly 17,298 rows are present.

Each row is a JSON object with a `messages` list of system, user, and assistant turns. The notebook applies the Qwen chat template before supervised fine-tuning.


## Dataset contents

The dataset is built by `modules/llm/lora_dataset/`:

```bash
cd modules/llm
python -m lora_dataset.create_dataset
```

The pipeline writes `data/processed/final_dataset/research_lora_train.jsonl`. The final merge uses shuffle seed 42 and combines:

- 13,992 open academic and ResearchQA rows from QASPER, SciTLDR, SciCite, PeerRead, and ResearchQA.
- 3,000 project arXiv RAG instruction rows from a 3,000-paper arXiv metadata sample.
- 6 local fixed-prompt seed rows that anchor project output formats.

Rows by source in the default dataset: `qasper` 3,000; `scitldr` 1,992; `scicite` 3,000; `peerread` 3,000; `researchqa` 3,000; `project_arxiv_rag` 3,000; `local_fixed_prompt` 6.


## Run order

1. Open the notebook in Google Colab or local Jupyter with an L4-class GPU or better.
2. Run **Install training packages**, **Imports**, and **Detect Colab**.
3. Run **Mount Google Drive on Colab** when training in Colab.
4. Run **Training configuration** and confirm the public dataset URL points at the 17,298-row JSONL.
5. Run **Resolve train data file** and **Validate train file format**.
6. Run the remaining cells in order. Checkpoints save every 50 steps; the notebook keeps the latest three Drive checkpoints and resumes automatically.
7. If a matching completion marker already exists, later cells skip to avoid accidental retraining. Set `FORCE_RETRAIN = True` only when you intend to archive the completed run and start again.


In [1]:
# Install training packages. Uses latest releases from pip; notebook expects current TRL SFTConfig API.
%pip install -q transformers peft trl accelerate datasets torch sentencepiece bitsandbytes gdown


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.9 MB/s eta 0:00:00


## Imports

Load standard libraries and training libraries in the next cell.


In [2]:
# Import path and file utilities.
import hashlib  # identify the exact training dataset used by a checkpoint
import json  # read JSONL rows and checkpoint metadata
import math  # compute epoch step counts and perplexity
import os  # detect Colab environment variable
import re  # parse checkpoint step numbers
import shutil  # copy checkpoints and zip adapter folder
import sys  # check for google.colab module
import uuid  # create atomic-copy temporary paths
from datetime import datetime, timezone  # build artifact timestamps
from pathlib import Path  # path joins for adapter output

# Import Hugging Face training stack.
import torch  # dtype for 4 bit compute
from datasets import load_dataset  # load JSONL as HF dataset
from peft import LoraConfig  # LoRA adapter configuration
from transformers import (  # model, tokenizer, training callback
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)
from trl import SFTConfig, SFTTrainer  # supervised fine tuning trainer


## Detect Colab and optional Drive mount

The next cell sets flags for Colab and whether Drive is mounted.


In [3]:
# True when running inside Google Colab.
IN_COLAB = "google.colab" in sys.modules

# Set True after you mount Drive in the next cell.
DRIVE_MOUNTED = False

# Drive folder for adapter zip and checkpoint backups.
DRIVE_ROOT = Path("/content/drive/MyDrive/colab_notebooks/nlp/research_lora_adapter")

print("Colab:", IN_COLAB)
print("Drive mounted flag (update after mount):", DRIVE_MOUNTED)


Colab: True
Drive mounted flag (update after mount): False


## Mount Google Drive on Colab

Run this cell before training. Cross-session resume requires Drive to remain mounted whenever a 50-step checkpoint is mirrored.


In [4]:
# Mount Drive only on Colab. Skip this cell when running locally.
if IN_COLAB:
    from google.colab import drive  # Colab Drive mount helper

    drive.mount("/content/drive")  # ask for permission once
    DRIVE_MOUNTED = True  # allow checkpoint copy to Drive
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)  # create backup root
    print("Drive root:", DRIVE_ROOT)
else:
    print("Not Colab. Skipping Drive mount.")


Mounted at /content/drive
Drive root: /content/drive/MyDrive/colab_notebooks/nlp/research_lora_adapter


## Training configuration

Training defaults to one full pass over the training split from the complete merged JSONL.

Train / val / test are split **inside this notebook** from the merged JSONL (80/10/10, seed 42).

Note: TRL 0.20+ uses `max_length` in `SFTConfig` (not `max_seq_length`).


In [5]:
# Hugging Face base model id. The saved adapter is later applied on top of Ollama qwen3:8b.
BASE_MODEL = "Qwen/Qwen3-8B"

# Stable key shared by every Colab session for this training run.
RUN_KEY = "qwen3_8b_lora_v2"

# Timestamp is used only for final adapter archives and deliberate retrain archives.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
FORCE_RETRAIN = False

JSONL_NAME = "research_lora_train.jsonl"
PUBLIC_TRAIN_JSONL_URL = "https://drive.google.com/file/d/1DDeI6ECLizmnVQGpnqXvuUWo7MY-XmLj/view?usp=sharing"
EXPECTED_FULL_DATASET_ROWS = 17298
MIN_FULL_DATASET_ROWS = 17298

if IN_COLAB:
    WORKSTREAM_ROOT = Path("/content")
    TRAIN_EXTRACT_DIR = Path("/content/train_data")
    TRAIN_JSONL = TRAIN_EXTRACT_DIR / JSONL_NAME
    FINAL_DATASET_DIR = None
    ADAPTER_ROOT = Path("/content/adapter")
else:
    WORKSTREAM_ROOT = Path.cwd().resolve()
    if (WORKSTREAM_ROOT / "training").is_dir():
        pass
    elif (WORKSTREAM_ROOT / "notebooks").is_dir() and (WORKSTREAM_ROOT.parent / "training").is_dir():
        WORKSTREAM_ROOT = WORKSTREAM_ROOT.parent
    elif not (WORKSTREAM_ROOT / "data").is_dir():
        WORKSTREAM_ROOT = WORKSTREAM_ROOT.parent
    FINAL_DATASET_DIR = WORKSTREAM_ROOT / "data/processed/final_dataset"
    TRAIN_EXTRACT_DIR = FINAL_DATASET_DIR
    TRAIN_JSONL = TRAIN_EXTRACT_DIR / JSONL_NAME
    ADAPTER_ROOT = WORKSTREAM_ROOT / "adapter"

# Stable local and Drive folders allow another Colab session to continue the run.
ADAPTER_DIR = ADAPTER_ROOT / "research_lora_adapter"
TRAINER_OUTPUT_DIR = ADAPTER_ROOT / "trainer_scratch" / RUN_KEY
LOCAL_RUN_METADATA_PATH = TRAINER_OUTPUT_DIR / "run_metadata.json"
DRIVE_RUN_DIR = DRIVE_ROOT / "checkpoints" / RUN_KEY
DRIVE_RUN_METADATA_PATH = DRIVE_RUN_DIR / "run_metadata.json"
COMPLETION_MARKER_PATH = DRIVE_RUN_DIR / "completed.json"
ZIP_PATH = ADAPTER_ROOT / f"research_lora_adapter_{RUN_ID}.zip"
METRICS_PATH = ADAPTER_DIR / "split_eval_metrics.json"

# LoRA hyperparameters (defaults for research adapter).
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 4096
PER_DEVICE_BATCH = 1
GRAD_ACCUM_STEPS = 8

# Train / val / test split (applied in notebook from merged JSONL).
SPLIT_SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1

# Full training and checkpoint policy.
NUM_TRAIN_EPOCHS = 1
EVALS_PER_EPOCH = 5
CHECKPOINT_SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 2
DRIVE_CHECKPOINTS_TO_KEEP = 3
LOGGING_STEPS = 5
EVAL_MAX_SAMPLES = 200
FINAL_EVAL_MAX_SAMPLES = 500

for folder in (ADAPTER_ROOT, ADAPTER_DIR, TRAINER_OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Colab:", IN_COLAB)
print("Stable run key:", RUN_KEY)
print("Train JSONL target:", TRAIN_JSONL)
print("Public train JSONL URL:", PUBLIC_TRAIN_JSONL_URL)
print("Expected dataset rows:", EXPECTED_FULL_DATASET_ROWS)
print("Checkpoint every:", CHECKPOINT_SAVE_STEPS, "optimizer steps")
print("Drive checkpoints retained:", DRIVE_CHECKPOINTS_TO_KEEP)
print("Force retrain:", FORCE_RETRAIN)


Colab: True
Stable run key: qwen3_8b_lora_v2
Train JSONL target: /content/train_data/research_lora_train.jsonl
Public train JSONL URL: https://drive.google.com/file/d/1DDeI6ECLizmnVQGpnqXvuUWo7MY-XmLj/view?usp=sharing
Expected dataset rows: 17298
Checkpoint every: 50 optimizer steps
Drive checkpoints retained: 3
Force retrain: False


## Resolve train data file

Run this before validation. It downloads the configured public Google Drive JSONL link to the runtime.

On Colab, the public JSONL downloads directly to `/content/train_data/research_lora_train.jsonl`. Local Jupyter also downloads from the same public link into the local `data/processed/final_dataset/` path.


In [6]:
def _download_public_train_jsonl():
    """Download the public Google Drive training JSONL into this runtime."""
    dest = TRAIN_JSONL
    dest.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading train JSONL from public Google Drive link:", PUBLIC_TRAIN_JSONL_URL)
    try:
        import gdown
    except ImportError as exc:
        raise ImportError("Install gdown first: %pip install gdown") from exc

    try:
        output = gdown.download(PUBLIC_TRAIN_JSONL_URL, str(dest), quiet=False, fuzzy=True)
    except TypeError:
        output = gdown.download(PUBLIC_TRAIN_JSONL_URL, str(dest), quiet=False)

    if output is None or not dest.is_file():
        raise FileNotFoundError(
            "Download failed. Confirm the Google Drive file is public and points to " + JSONL_NAME + "."
        )
    print("Downloaded train JSONL:", dest)
    return dest


def resolve_train_jsonl():
    return _download_public_train_jsonl()


TRAIN_JSONL = resolve_train_jsonl()
train_path = Path(TRAIN_JSONL)


Downloading...
From: https://drive.google.com/uc?id=1DDeI6ECLizmnVQGpnqXvuUWo7MY-XmLj
To: /content/train_data/research_lora_train.jsonl
100%|██████████| 65.9M/65.9M [00:00<00:00, 199MB/s]

Downloaded train JSONL: /content/train_data/research_lora_train.jsonl


## Validate train file and build resume identity

This cell validates the exact row count and computes the SHA-256 used to reject incompatible checkpoints.


In [7]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if not train_path.is_file():
    raise FileNotFoundError(f"Train JSONL not found: {train_path}")

with train_path.open("r", encoding="utf-8") as handle:
    first_line = handle.readline().strip()
    train_file_rows = 1 + sum(1 for _ in handle) if first_line else 0
if train_file_rows != EXPECTED_FULL_DATASET_ROWS:
    raise ValueError(
        f"Expected exactly {EXPECTED_FULL_DATASET_ROWS} rows; found {train_file_rows} "
        f"at {train_path}. Confirm the public JSONL link points to the updated dataset."
    )
row = json.loads(first_line)
messages = row.get("messages")
if not isinstance(messages, list) or len(messages) < 2:
    raise ValueError("First row must include messages with at least two turns.")
roles = {m.get("role") for m in messages}
if "user" not in roles or "assistant" not in roles:
    raise ValueError("messages must include user and assistant roles.")

TRAIN_DATASET_SHA256 = sha256_file(train_path)
CURRENT_RUN_METADATA = {
    "run_key": RUN_KEY,
    "base_model": BASE_MODEL,
    "dataset_rows": train_file_rows,
    "dataset_sha256": TRAIN_DATASET_SHA256,
    "split_seed": SPLIT_SEED,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "per_device_batch": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
}

print("Train file OK. Rows:", train_file_rows)
print("Dataset SHA256:", TRAIN_DATASET_SHA256)
print("Roles in first row:", sorted(roles))


Train file OK. Rows: 17298
Dataset SHA256: a690e624819fac57b80f6c38837e967bcf3173e222c363cfb56bdaed67b4eef9
Roles in first row: ['assistant', 'system', 'user']


## Load tokenizer

Qwen3 uses `trust_remote_code`. Pad token falls back to eos if missing.


In [8]:
# Load tokenizer from the base model hub id.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded. Vocab size:", len(tokenizer))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Tokenizer loaded. Vocab size: 151669


## Load base model with 4 bit QLoRA

This loads Qwen3 8B in 4 bit NF4 for GPU memory savings.


In [9]:
# 4 bit quantization config for QLoRA training.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load causal LM with automatic device map across GPU.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded with 4 bit quantization.")


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded with 4 bit quantization.


## Split train / val / test and build datasets

Loads the merged JSONL, shuffles with `SPLIT_SEED`, splits 80/10/10, and formats each split with the chat template.


In [10]:
def to_text(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text}


raw_dataset = load_dataset("json", data_files=str(train_path), split="train")
merged_rows = len(raw_dataset)
if merged_rows != EXPECTED_FULL_DATASET_ROWS:
    raise ValueError(
        f"Expected exactly {EXPECTED_FULL_DATASET_ROWS} rows; found {merged_rows} "
        f"at {train_path}."
    )
print(f"Merged rows: {merged_rows} (expected default: {EXPECTED_FULL_DATASET_ROWS})")

shuffled = raw_dataset.shuffle(seed=SPLIT_SEED)
total = len(shuffled)
train_end = int(total * TRAIN_RATIO)
val_end = train_end + int(total * VAL_RATIO)

train_raw = shuffled.select(range(train_end))
val_raw = shuffled.select(range(train_end, val_end))
test_raw = shuffled.select(range(val_end, total))

train_dataset = train_raw.map(to_text, remove_columns=train_raw.column_names)
val_dataset = val_raw.map(to_text, remove_columns=val_raw.column_names)
test_dataset = test_raw.map(to_text, remove_columns=test_raw.column_names)

print(
    f"Split (seed={SPLIT_SEED}, {TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{1 - TRAIN_RATIO - VAL_RATIO:.0%}):"
)
print("  train:", len(train_dataset))
print("  val:  ", len(val_dataset))
print("  test: ", len(test_dataset))

EFFECTIVE_BATCH_SIZE = PER_DEVICE_BATCH * GRAD_ACCUM_STEPS
STEPS_PER_EPOCH = math.ceil(len(train_dataset) / EFFECTIVE_BATCH_SIZE)
TOTAL_TRAINING_STEPS = STEPS_PER_EPOCH * NUM_TRAIN_EPOCHS
EVAL_STEPS = max(1, TOTAL_TRAINING_STEPS // EVALS_PER_EPOCH)
SAVE_STEPS = CHECKPOINT_SAVE_STEPS

print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print("Expected optimizer steps:", TOTAL_TRAINING_STEPS)
print("Eval every:", EVAL_STEPS, "steps | save every:", SAVE_STEPS, "steps")


Generating train split: 0 examples [00:00, ? examples/s]

Merged rows: 17298 (expected default: 17298)


Map:   0%|          | 0/13838 [00:00<?, ? examples/s]

Map:   0%|          | 0/1729 [00:00<?, ? examples/s]

Map:   0%|          | 0/1731 [00:00<?, ? examples/s]

Split (seed=42, 80%/10%/10%):
  train: 13838
  val:   1729
  test:  1731
Effective batch size: 8
Expected optimizer steps: 1730
Eval every: 346 steps | save every: 50 steps


## LoRA configuration

Small rank adapters on attention projections only.


In [11]:
# PEFT LoRA settings for causal language modeling.
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
print("LoRA config ready. rank:", LORA_R)


LoRA config ready. rank: 16


## Training configuration and resumable checkpoint mirror

The trainer saves locally every 50 optimizer steps. Each complete checkpoint is copied atomically to the stable Drive run folder, where the latest three valid checkpoints are retained.


In [12]:
CHECKPOINT_RE = re.compile(r"^checkpoint-(\d+)$")
REQUIRED_CHECKPOINT_FILES = ("trainer_state.json", "optimizer.pt", "scheduler.pt")
MODEL_FILE_NAMES = (
    "adapter_model.safetensors",
    "adapter_model.bin",
    "model.safetensors",
    "pytorch_model.bin",
)


def checkpoint_step(path: Path):
    match = CHECKPOINT_RE.fullmatch(path.name)
    return int(match.group(1)) if match else None


def is_valid_checkpoint(path: Path) -> bool:
    if not path.is_dir() or checkpoint_step(path) is None:
        return False
    if any(not (path / name).is_file() for name in REQUIRED_CHECKPOINT_FILES):
        return False
    if not any((path / name).is_file() for name in MODEL_FILE_NAMES):
        return False
    return (path / "rng_state.pth").is_file() or any(path.glob("rng_state_*.pth"))


def valid_checkpoints(parent: Path):
    if not parent.is_dir():
        return []
    paths = [path for path in parent.iterdir() if is_valid_checkpoint(path)]
    return sorted(paths, key=lambda path: checkpoint_step(path))


def latest_valid_checkpoint(parent: Path):
    paths = valid_checkpoints(parent)
    return paths[-1] if paths else None


def atomic_copytree(source: Path, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.parent / f".{destination.name}.partial-{uuid.uuid4().hex}"
    shutil.copytree(source, temporary)
    if destination.exists():
        shutil.rmtree(destination)
    temporary.replace(destination)
    return destination


def write_json_atomic(path: Path, payload: dict) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.partial-{uuid.uuid4().hex}")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    temporary.replace(path)
    return path


def load_json_object(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Expected JSON object in {path}")
    return payload


def metadata_mismatches(actual: dict, expected: dict):
    return [
        f"{key}: checkpoint={actual.get(key)!r}, current={value!r}"
        for key, value in expected.items()
        if actual.get(key) != value
    ]


def require_matching_metadata(path: Path) -> dict:
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint metadata missing: {path}")
    actual = load_json_object(path)
    mismatches = metadata_mismatches(actual, CURRENT_RUN_METADATA)
    if mismatches:
        raise ValueError("Checkpoint metadata mismatch:\n- " + "\n- ".join(mismatches))
    return actual


def prune_drive_checkpoints() -> None:
    checkpoints = valid_checkpoints(DRIVE_RUN_DIR)
    for old_checkpoint in checkpoints[:-DRIVE_CHECKPOINTS_TO_KEEP]:
        shutil.rmtree(old_checkpoint)
        print("Removed old Drive checkpoint:", old_checkpoint.name)


def completion_matches() -> bool:
    if not COMPLETION_MARKER_PATH.is_file():
        return False
    completion = load_json_object(COMPLETION_MARKER_PATH)
    return not metadata_mismatches(completion, CURRENT_RUN_METADATA)


if IN_COLAB and not DRIVE_MOUNTED:
    raise RuntimeError("Mount Google Drive before configuring resumable training.")

if FORCE_RETRAIN and IN_COLAB and DRIVE_RUN_DIR.exists():
    archive_dir = DRIVE_RUN_DIR.with_name(f"{RUN_KEY}_archive_{RUN_ID}")
    DRIVE_RUN_DIR.replace(archive_dir)
    print("Archived previous Drive run:", archive_dir)

TRAINING_ALREADY_COMPLETE = (
    IN_COLAB and DRIVE_MOUNTED and completion_matches() and not FORCE_RETRAIN
)
RESUME_CHECKPOINT = None
RESTORED_GLOBAL_STEP = 0

if not TRAINING_ALREADY_COMPLETE:
    write_json_atomic(LOCAL_RUN_METADATA_PATH, CURRENT_RUN_METADATA)
    if IN_COLAB and DRIVE_MOUNTED:
        DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
        drive_latest = latest_valid_checkpoint(DRIVE_RUN_DIR)
        if drive_latest is not None:
            require_matching_metadata(DRIVE_RUN_METADATA_PATH)
            restored = TRAINER_OUTPUT_DIR / drive_latest.name
            atomic_copytree(drive_latest, restored)
            if not is_valid_checkpoint(restored):
                raise RuntimeError(f"Restored checkpoint is incomplete: {restored}")
            RESUME_CHECKPOINT = restored
            trainer_state = load_json_object(restored / "trainer_state.json")
            RESTORED_GLOBAL_STEP = int(trainer_state.get("global_step", checkpoint_step(restored)))
        else:
            write_json_atomic(DRIVE_RUN_METADATA_PATH, CURRENT_RUN_METADATA)

sft_config = SFTConfig(
    output_dir=str(TRAINER_OUTPUT_DIR),
    max_steps=-1,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=CHECKPOINT_SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    dataset_text_field="text",
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to=[],
)


def mirror_checkpoint(checkpoint_dir: Path) -> None:
    if not is_valid_checkpoint(checkpoint_dir):
        raise RuntimeError(f"Trainer produced an incomplete checkpoint: {checkpoint_dir}")
    if IN_COLAB and DRIVE_MOUNTED:
        write_json_atomic(DRIVE_RUN_METADATA_PATH, CURRENT_RUN_METADATA)
        drive_destination = DRIVE_RUN_DIR / checkpoint_dir.name
        atomic_copytree(checkpoint_dir, drive_destination)
        if not is_valid_checkpoint(drive_destination):
            raise RuntimeError(f"Drive checkpoint copy is incomplete: {drive_destination}")
        prune_drive_checkpoints()
        print("Drive checkpoint:", drive_destination)


class CheckpointMirrorCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        mirror_checkpoint(checkpoint_dir)
        return control


mirror_callback = CheckpointMirrorCallback()
remaining_steps = max(0, TOTAL_TRAINING_STEPS - RESTORED_GLOBAL_STEP)
print("SFT config output dir:", TRAINER_OUTPUT_DIR)
print("Training status:", "already complete" if TRAINING_ALREADY_COMPLETE else ("resuming" if RESUME_CHECKPOINT else "new run"))
print("Resume checkpoint:", RESUME_CHECKPOINT or "none")
print("Restored optimizer step:", RESTORED_GLOBAL_STEP)
print("Expected remaining optimizer steps:", remaining_steps)
print("Checkpoint every:", CHECKPOINT_SAVE_STEPS, "steps")


SFT config output dir: /content/adapter/trainer_scratch/qwen3_8b_lora_v2
Training status: new run
Expected optimizer steps: 1730


## Train or resume

Creates the trainer and resumes optimizer, scheduler, RNG, and adapter state from the latest compatible Drive checkpoint. A completed run is not repeated unless `FORCE_RETRAIN=True`.


In [13]:
trainer = None
if TRAINING_ALREADY_COMPLETE:
    print("Matching completion marker found. Training is skipped.")
else:
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=sft_config,
        processing_class=tokenizer,
        peft_config=peft_config,
        callbacks=[mirror_callback],
    )
    trainer.args.max_eval_samples = EVAL_MAX_SAMPLES
    trainer.train(
        resume_from_checkpoint=str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None
    )
    print("Training finished.")


Training finished. Latest valid checkpoint: checkpoint-1730


## Evaluate train / val / test

Reports eval loss on all three splits and saves metrics to `adapter/research_lora_adapter/split_eval_metrics.json`.


In [14]:
if TRAINING_ALREADY_COMPLETE:
    print("Evaluation skipped because this run is already complete.")
else:
    def prepared_eval_dataset(split_name, raw_ds):
        """Use tokenized datasets from SFTTrainer; test split is prepared on demand."""
        if split_name == "train":
            return trainer.train_dataset
        if split_name == "val":
            return trainer.eval_dataset
        text_ds = raw_ds.select_columns(["text"])
        eval_packing = getattr(trainer.args, "eval_packing", None)
        packing = trainer.args.packing if eval_packing is None else eval_packing
        return trainer._prepare_dataset(
            text_ds,
            trainer.processing_class,
            trainer.args,
            packing,
            None,
            split_name,
        )


    trainer.args.remove_unused_columns = False

    split_metrics = {}
    for split_name, split_ds in [
        ("train", train_dataset),
        ("val", val_dataset),
        ("test", test_dataset),
    ]:
        eval_ds = prepared_eval_dataset(split_name, split_ds)
        sample_n = min(FINAL_EVAL_MAX_SAMPLES, len(eval_ds))
        eval_subset = eval_ds.select(range(sample_n))
        metrics = trainer.evaluate(eval_dataset=eval_subset, metric_key_prefix=split_name)
        split_metrics[split_name] = metrics
        loss_key = f"{split_name}_loss"
        loss = metrics.get(loss_key, metrics.get("eval_loss"))
        perplexity_key = f"{split_name}_perplexity"
        perplexity = metrics.get(perplexity_key, metrics.get("eval_perplexity"))
        if perplexity is None and loss is not None:
            perplexity = math.exp(loss) if loss < 100 else float("inf")
        loss_text = f"{loss:.4f}" if loss is not None else "n/a"
        perplexity_text = f"{perplexity:.4f}" if perplexity is not None else "n/a"
        print(
            f"{split_name:5s}  loss={loss_text}  perplexity={perplexity_text}  ",
            f"eval_samples={sample_n}  split_size={len(eval_ds)}"
        )

    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    METRICS_PATH.write_text(json.dumps(split_metrics, indent=2), encoding="utf-8")
    print("Saved metrics:", METRICS_PATH)


train  loss=0.9326  perplexity=2.5412  eval_samples=500  split_size=13838
val    loss=0.9388  perplexity=2.5570  eval_samples=500  split_size=1729
test   loss=0.9446  perplexity=2.5717  eval_samples=500  split_size=1731
Saved metrics: /content/adapter/research_lora_adapter/split_eval_metrics.json


## Save final adapter

Writes PEFT weights and tokenizer files to `adapter/research_lora_adapter/`.


In [15]:
if TRAINING_ALREADY_COMPLETE:
    print("Final adapter save skipped because this run is already complete.")
else:
    trainer.save_model(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    print("Final adapter saved to:", ADAPTER_DIR)


Final adapter saved to: /content/adapter/research_lora_adapter


## Zip adapter for download

Creates `adapter/research_lora_adapter_{RUN_ID}.zip` in the session.


In [16]:
def zip_adapter_folder(adapter_dir: Path, zip_base: Path) -> Path:
    archive = shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=adapter_dir.parent,
        base_dir=adapter_dir.name,
    )
    return Path(archive)


zip_file = None
if TRAINING_ALREADY_COMPLETE:
    print("Adapter zip skipped because this run is already complete.")
else:
    zip_file = zip_adapter_folder(ADAPTER_DIR, ZIP_PATH.with_suffix(""))
    print("Zip created:", zip_file)


Zip created: /content/adapter/research_lora_adapter_20260609_114209.zip


## Download zip on Colab

On local Jupyter, the print shows the zip path. Download the file from the file browser.


In [17]:
# Trigger browser download on Colab when a new adapter was trained.
if IN_COLAB and zip_file is not None:
    from google.colab import files

    files.download(str(zip_file))
else:
    print("Browser download skipped.")


## Copy final adapter to Drive and mark completion

The completion marker is written only after the final adapter ZIP has been copied successfully to Drive. Future sessions with matching metadata then skip accidental repeat training.


In [18]:
if TRAINING_ALREADY_COMPLETE:
    print("Drive adapter backup skipped because the matching run is already complete.")
elif IN_COLAB and DRIVE_MOUNTED and zip_file is not None:
    drive_adapters = DRIVE_ROOT / "adapters"
    drive_adapters.mkdir(parents=True, exist_ok=True)
    drive_zip = drive_adapters / zip_file.name
    temporary_zip = drive_zip.with_name(f".{drive_zip.name}.partial-{uuid.uuid4().hex}")
    shutil.copy2(zip_file, temporary_zip)
    temporary_zip.replace(drive_zip)
    completion_payload = {
        **CURRENT_RUN_METADATA,
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "adapter_zip": str(drive_zip),
        "final_global_step": int(trainer.state.global_step),
    }
    write_json_atomic(COMPLETION_MARKER_PATH, completion_payload)
    print("Drive zip backup:", drive_zip)
    print("Completion marker:", COMPLETION_MARKER_PATH)
else:
    print("Drive backup skipped; completion marker was not written.")


Drive zip backup: MyDrive/colab_notebooks/nlp/research_lora_adapter/adapters/research_lora_adapter_20260609_114209.zip
Completion marker: MyDrive/colab_notebooks/nlp/research_lora_adapter/checkpoints/qwen3_8b_lora_v2/completed.json


## Inspect resumable run state

Shows the stable Drive run, latest valid checkpoint, completion marker, and newest adapter ZIP.


In [19]:
def latest_child(parent: Path, pattern: str):
    matches = sorted(parent.glob(pattern), key=lambda path: path.stat().st_mtime, reverse=True)
    return matches[0] if matches else None


print("Run key:", RUN_KEY)
print("Local trainer directory:", TRAINER_OUTPUT_DIR)
if IN_COLAB and DRIVE_MOUNTED:
    print("Drive run directory:", DRIVE_RUN_DIR)
    print("Valid Drive checkpoints:", [path.name for path in valid_checkpoints(DRIVE_RUN_DIR)])
    print("Latest valid Drive checkpoint:", latest_valid_checkpoint(DRIVE_RUN_DIR) or "none")
    print("Matching completion marker:", completion_matches())

latest_zip = latest_child(ADAPTER_ROOT, "research_lora_adapter_*.zip")
print("Latest local zip:", latest_zip or "none")


Run key: qwen3_8b_lora_v2
Latest valid checkpoint: checkpoint-1730
Latest adapter zip: research_lora_adapter_20260609_114209.zip
